# 🦜 Bird Species Identifier

Upload any bird image and all trained classical ML models (SVM, k-NN, Decision Tree, Random Forest, Logistic Regression) will identify the species using frozen ResNet-18 deep visual features.

**This notebook is fully self-contained** — it will automatically download the dataset, extract features, and train all models if they are not already present.

In [ ]:
# Check if running in Google Colab and set up environment cleanly
import os
import shutil
from pathlib import Path

if 'COLAB_RELEASE_TAG' in os.environ:
    # 1. Force change directory back to the root '/content'
    os.chdir('/content')
    
    # 2. Clean up any accidental nested clone directories to save space and resolve path confusion
    nested_path = Path('/content/Bird-Species-Classification-ML/Bird-Species-Classification-ML')
    if nested_path.exists():
        print("Cleaning up accidental nested git clone folders...")
        shutil.rmtree(nested_path, ignore_errors=True)
        
    # 3. Clone repository if it doesn't exist under /content
    if not os.path.exists('Bird-Species-Classification-ML'):
        print("Cloning Bird-Species-Classification-ML repository...")
        !git clone https://github.com/01329582993/Bird-Species-Classification-ML.git
        
    # 4. Change working directory to '/content/Bird-Species-Classification-ML'
    %cd /content/Bird-Species-Classification-ML
else:
    print("Running locally. Working directory:", os.getcwd())

In [ ]:
# Load pre-computed features and labels
# If features are missing or outdated, automatically re-extract with improved feature set
import os
import sys
import shutil
import numpy as np
import joblib
from pathlib import Path
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path("./processed_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
npz_file = DATA_DIR / "combined_hog_color_lbp.npz"

# ---- Version check: ensure all individual feature files exist ----
if npz_file.exists():
    _check = np.load(npz_file)
    hog_file = DATA_DIR / "hog_features.npz"
    if _check['X_train'].shape[1] != 512 or not hog_file.exists():
        print(f"[Version check] Missing sub-feature files or outdated cache detected. Re-extracting full feature set (HOG, Color, LBP, Classical, Deep)...")
        for _f in DATA_DIR.glob('*.npz'): _f.unlink()
        for _f in DATA_DIR.glob('*.npy'): _f.unlink()
    del _check

if not npz_file.exists():
    print("Feature files not found. Running automatic feature extraction pipeline with cropping...")
    import pandas as pd
    import kagglehub
    from tqdm.auto import tqdm
    from PIL import Image
    from skimage.feature import hog, local_binary_pattern
    import cv2
    
    # ---- Step 1: Download and prepare dataset ----
    def locate_metadata():
        for p in [Path("metadata_preprocessed.csv"), DATA_DIR / "metadata_preprocessed.csv"]:
            if p.exists():
                return p
        return None

    metadata_file = locate_metadata()
    dataset_preprocessed_dir = Path("dataset_20_species_preprocessed")

    if metadata_file is None or not dataset_preprocessed_dir.exists():
        print("  Downloading CUB-200-2011 dataset via Kaggle Hub...")
        download_path = Path(kagglehub.dataset_download("wenewone/cub2002011"))
        DATASET_ROOT = download_path / "CUB_200_2011"
        IMAGES_FOLDER = DATASET_ROOT / "images"
        
        SUBSET_DIR = Path("./dataset_20_species")
        if SUBSET_DIR.exists():
            shutil.rmtree(SUBSET_DIR)
        SUBSET_DIR.mkdir(parents=True, exist_ok=True)
        
        bd_keywords = [
            'Crow', 'Kingfisher', 'Hummingbird', 'Mallard', 'Warbler',
            'Towhee', 'Jay', 'Creeper', 'Waxwing', 'Cuckoo',
            'Thrush', 'Woodpecker', 'Wren', 'Vireo', 'Catbird',
            'Meadowlark', 'Blackbird', 'Gull', 'Tern', 'Pelican'
        ]
        species_folders = sorted([f for f in IMAGES_FOLDER.iterdir() if f.is_dir()])
        selected_species = []
        for folder in species_folders:
            if any(kw.lower() in folder.name.lower() for kw in bd_keywords):
                if folder not in selected_species:
                    selected_species.append(folder)
            if len(selected_species) == 20:
                break
        
        for species_path in selected_species:
            shutil.copytree(str(species_path), SUBSET_DIR / species_path.name)
        
        classes_df = pd.read_csv(DATASET_ROOT / "classes.txt", sep=r"\s+", names=["class_id", "class_name"])
        images_df = pd.read_csv(DATASET_ROOT / "images.txt", sep=r"\s+", names=["image_id", "image_path"])
        labels_df = pd.read_csv(DATASET_ROOT / "image_class_labels.txt", sep=r"\s+", names=["image_id", "class_id"])
        bboxes_df = pd.read_csv(DATASET_ROOT / "bounding_boxes.txt", sep=r"\s+", names=["image_id", "x", "y", "width", "height"])
        
        selected_names = [p.name for p in selected_species]
        dataset_info = images_df.merge(labels_df, on="image_id").merge(classes_df, on="class_id").merge(bboxes_df, on="image_id")
        dataset_info = dataset_info[dataset_info["class_name"].isin(selected_names)].copy().reset_index(drop=True)
        dataset_info["full_image_path"] = dataset_info["image_path"].apply(lambda p: SUBSET_DIR / p)
        
        sys.path.append(str(Path(".").resolve()))
        import importlib
        import src.preprocessing
        importlib.reload(src.preprocessing)
        from src.preprocessing import create_stratified_splits, preprocess_and_save_image
        
        split_metadata = create_stratified_splits(dataset_info, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, random_state=42)
        
        if dataset_preprocessed_dir.exists():
            shutil.rmtree(dataset_preprocessed_dir)
        dataset_preprocessed_dir.mkdir(parents=True, exist_ok=True)
        
        preprocessed_paths = []
        for idx, row in split_metadata.iterrows():
            dst_file = dataset_preprocessed_dir / Path(row["image_path"])
            bbox = (row["x"], row["y"], row["width"], row["height"])
            preprocess_and_save_image(row["full_image_path"], dst_file, target_size=(224, 224), bbox=bbox)
            preprocessed_paths.append(str(dst_file))
        split_metadata["preprocessed_image_path"] = preprocessed_paths
        cols = ["image_id", "image_path", "class_id", "class_name", "split", "preprocessed_image_path", "x", "y", "width", "height"]
        split_metadata[cols].to_csv("metadata_preprocessed.csv", index=False)
        split_metadata[cols].to_csv(DATA_DIR / "metadata_preprocessed.csv", index=False)
        metadata_file = Path("metadata_preprocessed.csv")
        print("  Dataset preparation complete!")

    # ---- Step 2: Feature Extractions (Classical + Deep ResNet18) ----
    import torch
    import torchvision.models as models_tv
    import torchvision.transforms as transforms_tv
    import cv2
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  Loading pre-trained ResNet18 feature extractor on {device}...")
    weights = models_tv.ResNet18_Weights.DEFAULT
    resnet = models_tv.resnet18(weights=weights)
    resnet.fc = torch.nn.Identity()  # Remove final FC layer -> 512-dim embedding
    resnet.eval()
    resnet.to(device)
    
    transform_deep = transforms_tv.Compose([
        transforms_tv.Resize((224, 224)),
        transforms_tv.ToTensor(),
        transforms_tv.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    def _extract_deep_features(img):
        tensor = transform_deep(img.convert("RGB")).unsqueeze(0).to(device)
        with torch.no_grad():
            feat = resnet(tensor).squeeze(0).cpu().numpy()
        return feat.astype(np.float32)

    def _extract_hog(img):
        arr = np.array(img.resize((128, 128), Image.Resampling.LANCZOS))
        gray = np.mean(arr, axis=2).astype(np.uint8)
        return hog(gray, orientations=9, pixels_per_cell=(8,8), cells_per_block=(2,2), block_norm='L2-Hys', visualize=False).astype(np.float32)

    def _extract_color(img):
        img_arr = np.array(img.convert("RGB"), dtype=np.uint8)
        hists = []
        for ch in range(3):
            h, _ = np.histogram(img_arr[:,:,ch], bins=32, range=(0,256))
            h = h.astype(np.float32); h /= (h.sum() + 1e-9)
            hists.append(h)
        return np.concatenate(hists)

    def _extract_lbp(img):
        img_arr = np.array(img.convert("L"), dtype=np.uint8)
        lbp = local_binary_pattern(img_arr, P=8, R=1, method="uniform")
        h, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 11), range=(0, 10))
        h = h.astype(np.float32); h /= (h.sum() + 1e-9)
        return h

    metadata_df = pd.read_csv(locate_metadata())
    X_deep, X_hog, X_color, X_lbp, y_all, splits_all = [], [], [], [], [], []

    print("  Extracting Deep + HOG + Color + LBP features for complete ablation study...")
    for _, row in tqdm(metadata_df.iterrows(), total=len(metadata_df)):
        img_path = Path(row["preprocessed_image_path"])
        img = Image.open(img_path).convert("RGB")
        X_deep.append(_extract_deep_features(img))
        X_hog.append(_extract_hog(img))
        X_color.append(_extract_color(img))
        X_lbp.append(_extract_lbp(img))
        y_all.append(row["class_id"] - 1)
        splits_all.append(row["split"])

    X_deep = np.array(X_deep, dtype=np.float32)
    X_hog = np.array(X_hog, dtype=np.float32)
    X_color = np.array(X_color, dtype=np.float32)
    X_lbp = np.array(X_lbp, dtype=np.float32)
    y_all = np.array(y_all, dtype=np.int32)
    splits_all = np.array(splits_all)
    X_classical = np.concatenate([X_hog, X_color, X_lbp], axis=1)

    unique_classes = sorted(metadata_df["class_name"].unique())
    lm = {cls: idx for idx, cls in enumerate(unique_classes)}
    joblib.dump(lm, DATA_DIR / "label_mapping.pkl")

    def _save_split_npz(fname, X):
        np.savez_compressed(
            DATA_DIR / fname,
            X_train=X[splits_all=="train"], y_train=y_all[splits_all=="train"],
            X_val=X[splits_all=="val"],   y_val=y_all[splits_all=="val"],
            X_test=X[splits_all=="test"],  y_test=y_all[splits_all=="test"]
        )
    _save_split_npz("hog_features.npz", X_hog)
    _save_split_npz("color_features.npz", X_color)
    _save_split_npz("lbp_features.npz", X_lbp)
    _save_split_npz("classical_combined.npz", X_classical)
    _save_split_npz("deep_features.npz", X_deep)
    _save_split_npz("combined_hog_color_lbp.npz", X_deep)
    print(f"  Feature extractions complete! Saved all individual split files.")

# ---- Load the features ----
data = np.load(npz_file)
X_train, y_train = data["X_train"], data["y_train"]
X_val,   y_val   = data["X_val"],   data["y_val"]
X_test,  y_test  = data["X_test"],  data["y_test"]

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

label_mapping = joblib.load(DATA_DIR / "label_mapping.pkl")
class_names = [k.split('.')[-1].replace('_', ' ') for k in sorted(label_mapping, key=label_mapping.get)]

print(f"Loaded feature dataset: {npz_file.name}")
print(f"  Training set  : {X_train.shape} (Standardized)")
print(f"  Validation set: {X_val.shape} (Standardized)")
print(f"  Testing set   : {X_test.shape} (Standardized)")
print(f"  Classes ({len(class_names)}): {class_names[:5]}...")

In [ ]:
# ---- Load or auto-train all ML models ----
import joblib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torchvision.models as tv_models
import torchvision.transforms as tv_transforms

MODELS_DIR = Path('./models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

model_files = {
    'SVM':                'svm_model.pkl',
    'k-NN':               'knn_model.pkl',
    'Decision Tree':      'decision_tree_model.pkl',
    'Random Forest':      'random_forest_model.pkl',
    'Logistic Regression':'logistic_regression_model.pkl'
}

models = {}
for name, fname in model_files.items():
    p = MODELS_DIR / fname
    if p.exists():
        loaded_m = joblib.load(p)
        exp_dims = getattr(loaded_m, 'n_features_in_', None)
        if exp_dims is not None and exp_dims != X_train.shape[1]:
            print(f'  [Outdated] {name} was trained on {exp_dims} dims, retraining on {X_train.shape[1]}...')
            p.unlink()
        else:
            models[name] = loaded_m
            print(f'  Loaded: {name}')

# Auto-train any missing models
if len(models) < len(model_files):
    from sklearn.svm import SVC
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    fallback = {
        'SVM':                SVC(C=10, kernel='rbf', random_state=42),
        'k-NN':               KNeighborsClassifier(n_neighbors=5, weights='distance'),
        'Decision Tree':      DecisionTreeClassifier(max_depth=20, random_state=42),
        'Random Forest':      RandomForestClassifier(n_estimators=200, max_depth=25, random_state=42, n_jobs=-1),
        'Logistic Regression':LogisticRegression(C=1.0, max_iter=2000, random_state=42)
    }
    for name, clf in fallback.items():
        if name not in models:
            print(f'  Training {name}...')
            clf.fit(X_train, y_train)
            models[name] = clf
            joblib.dump(clf, MODELS_DIR / model_files[name])
            print(f'    Saved: {name}')

print(f'\n✅ {len(models)} model(s) ready for bird identification!')

In [ ]:
# ---- Load frozen ResNet-18 feature extractor (re-fit scaler on training data) ----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

weights = tv_models.ResNet18_Weights.DEFAULT
resnet = tv_models.resnet18(weights=weights)
resnet.fc = torch.nn.Identity()
resnet.eval()
resnet.to(device)

transform = tv_transforms.Compose([
    tv_transforms.Resize((224, 224)),
    tv_transforms.ToTensor(),
    tv_transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
])

# Re-fit a scaler using the already-loaded X_train (from DATA_LOADER_CELL)
from sklearn.preprocessing import StandardScaler as _SC
deep_npz = DATA_DIR / 'deep_features.npz'
if deep_npz.exists():
    _raw = np.load(deep_npz)
    id_scaler = _SC().fit(_raw['X_train'])
else:
    id_scaler = _SC().fit(X_train)  # fallback: use already-standardized X_train as proxy
    print('[Note] Using standardized X_train as scaler fallback.')

def extract_deep_features(img):
    tensor = transform(img.convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = resnet(tensor).squeeze(0).cpu().numpy()
    return id_scaler.transform(feat.reshape(1, -1).astype('float32'))

print('ResNet-18 feature extractor ready!')

In [ ]:
# ---- Upload your bird image ----
from google.colab import files as colab_files

print('Please upload a bird image (JPG/PNG)...')
uploaded = colab_files.upload()

if not uploaded:
    raise ValueError('No file uploaded. Please upload a bird image.')

img_filename = list(uploaded.keys())[0]
img = Image.open(img_filename).convert('RGB')

plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.title(f'Uploaded Image: {img_filename}', fontsize=12)
plt.axis('off')
plt.tight_layout()
plt.show()
print(f'Image size: {img.size}')

In [ ]:
# ---- Predict bird species using all trained ML models ----
features = extract_deep_features(img)

predictions = {}
probabilities = {}

for model_name, clf in models.items():
    pred_idx = clf.predict(features)[0]
    pred_species = class_names[pred_idx] if pred_idx < len(class_names) else f'Unknown (class {pred_idx})'
    predictions[model_name] = pred_species
    if hasattr(clf, 'predict_proba'):
        proba = clf.predict_proba(features)[0]
        probabilities[model_name] = round(float(np.max(proba)) * 100, 1)
    else:
        probabilities[model_name] = None

print('\n' + '='*55)
print('         BIRD SPECIES IDENTIFICATION RESULTS')
print('='*55)
for model_name, species in predictions.items():
    conf = f'  (Confidence: {probabilities[model_name]:.1f}%)' if probabilities[model_name] else ''
    print(f'  {model_name:<22}: {species}{conf}')
print('='*55)

from collections import Counter
majority_species = Counter(predictions.values()).most_common(1)[0][0]
print(f'\n🏆 MAJORITY VOTE PREDICTION: {majority_species}')

In [ ]:
# ---- Visualize: uploaded image + color-coded model predictions ----
model_names    = list(predictions.keys())
species_preds  = list(predictions.values())
unique_species = sorted(set(species_preds))

cmap = plt.cm.get_cmap('tab10', len(unique_species))
species_colors = {sp: cmap(i) for i, sp in enumerate(unique_species)}
bar_colors     = [species_colors[s] for s in species_preds]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(img)
axes[0].set_title('Uploaded Bird Image', fontsize=13, fontweight='bold')
axes[0].axis('off')

y_pos = np.arange(len(model_names))
bars  = axes[1].barh(y_pos, [1]*len(model_names), color=bar_colors, edgecolor='gray')
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(model_names, fontsize=11)
axes[1].set_xticks([])
axes[1].set_title('Model Predictions', fontsize=13, fontweight='bold')
axes[1].invert_yaxis()

for i, (bar, sp) in enumerate(zip(bars, species_preds)):
    conf_text = f' ({probabilities[model_names[i]]:.1f}%)' if probabilities[model_names[i]] else ''
    axes[1].text(0.02, bar.get_y() + bar.get_height()/2,
                 f'{sp}{conf_text}', va='center', fontsize=10, fontweight='bold')

patches = [mpatches.Patch(color=species_colors[sp], label=sp) for sp in unique_species]
axes[1].legend(handles=patches, loc='lower right', fontsize=9, title='Predicted Species')

plt.suptitle(f'🏆 Majority Vote: {majority_species}', fontsize=15, fontweight='bold', color='darkgreen')
plt.tight_layout()
plt.show()